In [107]:
import pandas as pd 
from pathlib import Path
import re 
import json 
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from sklearn.decomposition import PCA
import seaborn as sns  # kept for color palette only

In [108]:
d = Path("/home/finn/workspace/creatures/logs/")
assert d.exists()

In [109]:
d = sorted(list(d.glob("*.log")))
p = d[-1]

In [110]:
with open(p) as f:
    n_lines = sum(1 for _ in f)
print(n_lines)

69267


In [111]:
_header_pat = re.compile(
    r"\[simulation_start_ts=(?P<simulation_start_ts>\d+)\] "
    r"\[unix_ts=(?P<unix_timestamp>\d+)\] "
    r"\[frame=(?P<frame>\d+)\] "
    r"\[level=INFO\] "
    r"animal_despawn "
    r"reason=(?P<reason>\w+) "
    r"lifetime_frames=(?P<lifetime>\d+) "
    r"animal=Animal\s*\{(?P<animal>.+)\}$"
)

def _split_top_level(text):
    """Split on commas not inside any bracket/brace/paren."""
    depth, parts, current = 0, [], []
    for ch in text:
        if ch in "({[":
            depth += 1
        elif ch in ")}]":
            depth -= 1
        elif ch == "," and depth == 0:
            parts.append("".join(current).strip())
            current = []
            continue
        current.append(ch)
    if current:
        parts.append("".join(current).strip())
    return parts

def _coerce(val):
    if val == "None":
        return None
    m = re.match(r"^Some\((.+)\)$", val)
    if m:
        return _coerce(m.group(1))
    try:
        return int(val)
    except ValueError:
        pass
    try:
        return float(val)
    except ValueError:
        pass
    return val

def extract_animal_data(line):
    m = _header_pat.match(line.rstrip())
    if m is None:
        return None

    animal = {}
    for part in _split_top_level(m.group("animal")):
        key, sep, val = part.partition(":")
        if sep:
            animal[key.strip()] = _coerce(val.strip())

    return {
        "simulation_start_ts": int(m.group("simulation_start_ts")),
        "unix_timestamp": float(m.group("unix_timestamp")),
        "lifetime": float(m.group("lifetime")),
        "reason": m.group("reason"),
        **animal,
    }


In [112]:
with open(p) as f:
    df = pd.DataFrame([r for line in f if (r := extract_animal_data(line)) is not None])

In [113]:
df

,simulation_start_ts,unix_timestamp,lifetime,reason,id,parent_id,diet,position,velocity,energy,initial_energy,size,color,vision,genome,spawn_at,despawn_at,family,generation
0,1782982134,1.782982e+09,1204.0,starvation,16679461234484274892,NaN,Herbivore,"Vec2(-567.9771, -168.152)","Vec2(-2.3867002, -59.354416)",0.0,50.0,10.0,"Srgba(Srgba { red: 0.0, green: 0.8, blue: 0.0,...",Vision { range: 180.0 },"Genome { genes: [-0.39495063, -0.35410357, 0.7...",423,1627,3715340143,0
1,1782982134,1.782982e+09,1393.0,starvation,4868339113372778220,NaN,Carnivore,"Vec2(296.667, 229.48296)","Vec2(-37.802334, 36.609867)",0.0,50.0,10.0,"Srgba(Srgba { red: 1.0, green: 0.0, blue: 0.0,...",Vision { range: 180.0 },"Genome { genes: [-1.2413747, -1.8102806, 0.494...",939,2332,311097734,3
2,1782982134,1.782982e+09,2506.0,starvation,12439945921596447173,NaN,Carnivore,"Vec2(-103.626465, 319.32944)","Vec2(12.443382, -6.112447)",0.0,50.0,10.0,"Srgba(Srgba { red: 1.0, green: 0.0, blue: 0.0,...",Vision { range: 180.0 },"Genome { genes: [1.8951735, 2.424253, 0.295581...",1,2507,4025200342,7
3,1782982134,1.782982e+09,2119.0,starvation,268049991015687437,NaN,Carnivore,"Vec2(-190.70949, 105.47633)","Vec2(-24.935898, -49.237648)",0.0,50.0,10.0,"Srgba(Srgba { red: 1.0, green: 0.0, blue: 0.0,...",Vision { range: 180.0 },"Genome { genes: [-0.20172262, 1.4517004, -0.68...",1155,3274,1450608125,2
4,1782982134,1.782982e+09,2565.0,starvation,6235895345676587433,NaN,Carnivore,"Vec2(332.77518, 114.300285)","Vec2(12.322927, 5.132164)",0.0,50.0,10.0,"Srgba(Srgba { red: 1.0, green: 0.0, blue: 0.0,...",Vision { range: 180.0 },"Genome { genes: [-0.12586951, -0.9911046, 0.77...",778,3343,113814507,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6355,1782982134,1.782989e+09,272.0,collision,486501693314361062,NaN,Herbivore,"Vec2(413.35474, 348.49957)","Vec2(-7.9179153, 1.228415)",0.0,50.0,10.0,"Srgba(Srgba { red: 0.0, green: 0.8, blue: 0.0,...",Vision { range: 180.0 },"Genome { genes: [-0.39495063, -0.35410357, 0.7...",235928,236200,3715340143,0
6356,1782982134,1.782989e+09,2557.0,collision,11572706215316957510,1.783705e+18,Herbivore,"Vec2(333.56332, 264.0629)","Vec2(112.58937, 26.623562)",0.0,50.0,10.0,"Srgba(Srgba { red: 0.0, green: 0.8, blue: 0.0,...",Vision { range: 180.0 },"Genome { genes: [-2.548998, -5.795806, -1.3543...",233649,236206,20251590,119
6357,1782982134,1.782989e+09,105.0,collision,9067298079714557882,NaN,Herbivore,"Vec2(620.554, -311.5974)","Vec2(10.653208, -4.9192743)",0.0,50.0,10.0,"Srgba(Srgba { red: 0.0, green: 0.8, blue: 0.0,...",Vision { range: 180.0 },"Genome { genes: [-0.013694525, -0.18126392, 0....",236132,236237,89846771,0
6358,1782982134,1.782989e+09,646.0,collision,13272327214195063229,1.783705e+18,Herbivore,"Vec2(551.008, -104.55174)","Vec2(-34.46175, 13.025267)",0.0,50.0,10.0,"Srgba(Srgba { red: 0.0, green: 0.8, blue: 0.0,...",Vision { range: 180.0 },"Genome { genes: [-2.5341032, -5.2494164, -1.36...",235593,236239,20251590,119


In [114]:
_genome_pat = re.compile(r"Genome \{ genes: \[(.+?)\] \}")

def parse_genome(s):
    m = _genome_pat.search(s)
    return np.fromstring(m.group(1), sep=", ") if m else None

df["genome"] =df["genome"].map(parse_genome)

In [115]:
df["family"]

0       3715340143
1        311097734
2       4025200342
3       1450608125
4        113814507
           ...    
6355    3715340143
6356      20251590
6357      89846771
6358      20251590
6359      20251590
Name: family, Length: 6360, dtype: int64

In [116]:
from sklearn.decomposition import PCA
X = np.stack(df["genome"].to_list())
pca = PCA(n_components=2)
X_transformed  = pca.fit_transform(X)

In [117]:
unique_families = np.sort(df["family"].dropna().unique())
palette = sns.color_palette("husl", n_colors=len(unique_families))

family_to_color = {fam: palette[i] for i, fam in enumerate(unique_families)}
family_colors = df["family"].map(family_to_color)

In [118]:
family_colors.iloc[0]

(0.9594780427173869, 0.37329627032445234, 0.8985363994093621)

In [119]:
color_discrete_map = {
    str(fam): f"rgb({int(r*255)},{int(g*255)},{int(b*255)})"
    for fam, (r, g, b) in family_to_color.items()
}

fig = px.scatter(
    data_frame=df, 
    x=X_transformed[:, 0],
    y=X_transformed[:, 1],
    color=df["family"].astype(str),
    hover_data=["family", "diet", "lifetime"],
    color_discrete_map=color_discrete_map,
    labels={"x": "PC1", "y": "PC2", "color": "Family"},
    title="Genome PCA by Family",
)
fig.show()

In [120]:
fig = px.bar(
    df.groupby("diet")["lifetime"].mean().reset_index(),
    x="diet",
    y="lifetime",
    color="diet",
    title="Mean Lifetime by Diet",
    labels={"lifetime": "mean lifetime (frames)"},
)
fig.show()

In [123]:
df_fam = df.sort_values(by="lifetime", ascending=False)[:1000]
agg = (
    df_fam.groupby("family")["lifetime"]
    .mean()
    .reset_index()
    .sort_values("lifetime", ascending=False)
)
agg["family"] = agg["family"].astype(str)

fig = px.bar(
    agg,
    x="family",
    y="lifetime",
    title="Mean Lifetime by Family (top 1000 longest-lived animals)",
    labels={"lifetime": "mean lifetime (frames)", "family": "family"},
)
fig.show()

# Population

In [ ]:
def extract_population_data(line):
    m = re.match(
        r"\[simulation_start_ts=(?P<simulation_start_ts>\d+)\] "
        r"\[unix_ts=(?P<unix_ts>\d+)\] "
        r"\[frame=(?P<frame>\d+)\] "
        r"\[level=INFO\] "
        r"population_size "
        r"plants=(?P<plants>\d+) "
        r"animals=\{(?P<animals>[^}]+)\} "
        r"families=(?P<families>\d+:\d+(?:\|\d+:\d+)*)",
        line,
    )
    if m is None:
        return None

    animals = {k: int(v) for k, v in (pair.split(":") for pair in m.group("animals").split())}
    families = [tuple(map(int, f.split(":"))) for f in m.group("families").split("|")]

    return {
        "simulation_start_ts": int(m.group("simulation_start_ts")),
        "unix_ts": int(m.group("unix_ts")),
        "frame": int(m.group("frame")),
        "n_plants": int(m.group("plants")),
        "carnivores": animals.get("carnivores", 0),
        "herbivores": animals.get("herbivores", 0),
        "omnivores": animals.get("omnivores", 0),
        "scavengers": animals.get("scavengers", 0),
        "families": families,
    }


In [ ]:
with open(p) as f:
    df = pd.DataFrame([r for line in f if (r := extract_population_data(line)) is not None]).set_index("frame")
df

,simulation_start_ts,unix_ts,n_plants,carnivores,herbivores,omnivores,scavengers,families
frame,,,,,,,,
27,1782982134,1782982135,11,1,0,0,0,"[(4025200342, 1)]"
39,1782982134,1782982135,12,1,0,0,0,"[(4025200342, 1)]"
158,1782982134,1782982137,13,1,0,0,0,"[(4025200342, 1)]"
240,1782982134,1782982139,14,1,0,0,0,"[(4025200342, 1)]"
325,1782982134,1782982140,15,1,0,0,0,"[(4025200342, 1)]"
...,...,...,...,...,...,...,...,...
236256,1782982134,1782989098,84,40,39,0,0,"[(20251590, 35), (89846771, 2), (561725692, 1)..."
236260,1782982134,1782989098,84,40,41,0,0,"[(20251590, 37), (89846771, 2), (561725692, 1)..."
236262,1782982134,1782989098,84,42,41,0,0,"[(20251590, 37), (89846771, 2), (561725692, 1)..."


### Genetic DIversity

In [ ]:
def f(x):
    return {"n_families": len(x), "largest_family_size": max(x, key=lambda i: i[1])[1]}
df_fam = df["families"].apply(f).apply(pd.Series)

fig = px.line(
    df_fam,
    title="Genetic Diversity over Time",
    labels={"index": "frame", "value": "count", "variable": "metric"},
)
fig.show()

### Population

In [ ]:
df

,simulation_start_ts,unix_ts,n_plants,carnivores,herbivores,omnivores,scavengers,families
frame,,,,,,,,
27,1782982134,1782982135,11,1,0,0,0,"[(4025200342, 1)]"
39,1782982134,1782982135,12,1,0,0,0,"[(4025200342, 1)]"
158,1782982134,1782982137,13,1,0,0,0,"[(4025200342, 1)]"
240,1782982134,1782982139,14,1,0,0,0,"[(4025200342, 1)]"
325,1782982134,1782982140,15,1,0,0,0,"[(4025200342, 1)]"
...,...,...,...,...,...,...,...,...
236256,1782982134,1782989098,84,40,39,0,0,"[(20251590, 35), (89846771, 2), (561725692, 1)..."
236260,1782982134,1782989098,84,40,41,0,0,"[(20251590, 37), (89846771, 2), (561725692, 1)..."
236262,1782982134,1782989098,84,42,41,0,0,"[(20251590, 37), (89846771, 2), (561725692, 1)..."


In [ ]:
fig = px.line(
    df[["n_plants", "carnivores", "herbivores", "omnivores", "scavengers"]],
    title="Population over Time",
    labels={"index": "frame", "value": "count", "variable": "species"},
)
fig.show()

In [ ]:
MAX_SPPED=30
EXPONENT=0.2

In [ ]:
x = np.arange(0, 1, 0.1)
fig = go.Figure()
for b in np.arange(1.0, 4, 0.5):
    a = 1.0
    y = x + a * (b**x) - a
    fig.add_trace(go.Scatter(x=x, y=y, mode="lines", name=f"b={b:.1f}"))

fig.update_layout(
    title="Speed Curves",
    xaxis_title="x",
    yaxis_title="y",
)
fig.show()